# Chapter 2 — Building a Simple Tokenizer from Scratch

This notebook builds a small word-and-punctuation tokenizer directly from a text corpus. The purpose is not to compete with production tokenizers; it is to make the core transformation visible: raw text is split into tokens, each distinct token receives an integer ID, and the mapping is used in both directions. These steps establish the vocabulary interface that a language model needs before token embeddings can be learned.

The workflow begins by downloading Edith Wharton's short story *The Verdict* from the official companion repository and reading it as UTF-8 text. Regular expressions are then refined incrementally so punctuation and whitespace become explicit boundaries without leaving empty tokens behind. The resulting corpus tokens define a sorted vocabulary and two lookup tables: token-to-ID for encoding and ID-to-token for decoding.

Two tokenizer versions reveal an important design constraint. `SimpleTokenizerV1` works only for tokens already observed in the corpus. Its preserved `KeyError: 'Hello'` is an intentional educational checkpoint, not a result to hide: it shows exactly what happens when an out-of-vocabulary token is encountered. `SimpleTokenizerV2` extends the vocabulary with `<|unk|>` for unseen tokens and `<|endoftext|>` for separating independent text segments.

## Learning objectives

By the end of the notebook, you should be able to:

- split text into words and punctuation with `re.split()`;
- remove whitespace-only and empty fragments safely;
- build deterministic forward and reverse vocabulary mappings;
- encode text as integer token IDs and decode IDs back to text;
- explain the failure mode of a closed vocabulary;
- use unknown and end-of-text tokens to handle broader inputs.

| Stage | Main representation | Purpose |
|---|---|---|
| Raw corpus | Python string | Source language data |
| Preprocessing | Token list | Separate words and punctuation |
| Vocabulary | Token ↔ integer mappings | Assign stable numerical IDs |
| Tokenizer V1 | Known-token IDs | Demonstrate the basic lookup design |
| Tokenizer V2 | IDs plus special tokens | Handle unseen words and text boundaries |


## 1. Load the source text

The first cells download `the-verdict.txt` from the book's official companion repository and save it locally as `text-verdict.txt`. The notebook then reads the file with UTF-8 encoding and prints a short preview.

The preserved output records **20,479 characters** in the downloaded text. Internet access is needed only when the download cell must create or replace the local file; later cells read the local copy.


In [1]:
import urllib.request

url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")
file_path = "text-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('text-verdict.txt', <http.client.HTTPMessage at 0x1de22271590>)

In [2]:
with open("text-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total Number of Character: ", len(raw_text))
print(raw_text[:99])

Total Number of Character:  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


## 2. Refine token boundaries with regular expressions

The preprocessing is developed in small, inspectable steps. Splitting only on whitespace leaves punctuation attached to words. Capturing punctuation and whitespace in the regular expression exposes those separators as tokens, after which empty strings and whitespace-only fragments can be removed.

The final pattern recognizes common punctuation marks, quotes, parentheses, the double hyphen, and whitespace:

```python
r'([.,:;?!_"()\']|--|\s)'
```

Because the separator is placed inside a capturing group, `re.split()` retains it in the result. That behavior is useful here: punctuation carries structure and receives its own vocabulary IDs instead of disappearing.


In [3]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [4]:
result = re.split(r'([.,]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [5]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [6]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([.,:;?!_"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


## 3. Tokenize the corpus and build a vocabulary

The complete story is split with the refined expression, normalized by stripping whitespace, and filtered to remove empty elements. Calling `set()` identifies unique tokens, while sorting makes the token-to-ID assignment deterministic.

The preserved outputs show:

- **4,690 token occurrences** after preprocessing;
- **1,130 unique vocabulary entries** before adding special tokens;
- punctuation occupying the first IDs because the vocabulary is sorted.

These counts describe the saved notebook execution and the current source file. They are not universal properties of the text-tokenization task.


In [7]:
preprocessed = re.split(r'([.,:;?!_"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


In [8]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [9]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [10]:
vocab = {token:integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


## 4. Implement `SimpleTokenizerV1`

The first tokenizer stores two synchronized dictionaries:

- `str_to_int` maps token strings to integer IDs;
- `int_to_str` reverses those IDs during decoding.

Encoding repeats the same regular-expression preprocessing used to build the vocabulary, then performs a direct lookup for every token. Decoding joins token strings with spaces and removes artificial spaces before punctuation. The round trip is intentionally simple and makes the responsibilities of a tokenizer concrete.


In [11]:
class SimpleTokenizerV1:
    """Map corpus tokens to integer IDs without unknown-token handling.

    Args:
        vocab (dict[str, int]): Mapping from token strings to integer IDs.

    Attributes:
        str_to_int (dict[str, int]): Forward token-to-ID lookup.
        int_to_str (dict[int, str]): Reverse ID-to-token lookup.
    """

    def __init__(self, vocab):
        """Initialize forward and reverse vocabulary mappings."""
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        """Convert text into vocabulary IDs.

        Args:
            text (str): Text to split and encode.

        Returns:
            list[int]: Integer IDs for the preprocessed tokens.

        Raises:
            KeyError: If a token is absent from the vocabulary.
        """
        preprocessed = re.split(r'([.,:;?!_"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        """Convert vocabulary IDs back into readable text.

        Args:
            ids (list[int]): Token IDs to decode.

        Returns:
            str: Reconstructed text with punctuation spacing normalized.

        Raises:
            KeyError: If an ID is absent from the reverse vocabulary.
        """
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [12]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know, "
       Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [13]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


## 5. Observe the out-of-vocabulary failure

`SimpleTokenizerV1` has no policy for a token that was absent from its training corpus. Encoding “Hello, do you like tea?” therefore reaches a dictionary lookup for `Hello` and the preserved output records `KeyError: 'Hello'`.

This error is kept because it motivates the next design change. A practical tokenizer needs either a vocabulary capable of representing arbitrary text or an explicit fallback token. The notebook chooses the second option for this educational implementation.


In [14]:
text = "Hello, do you like tea?"
tokenizer.encode(text)

KeyError: 'Hello'

## 6. Add special tokens and implement `SimpleTokenizerV2`

The vocabulary is extended with two reserved entries:

- `<|unk|>` represents any token missing from the learned vocabulary;
- `<|endoftext|>` marks a boundary between otherwise separate documents or passages.

`SimpleTokenizerV2.encode()` replaces unseen tokens with `<|unk|>` before performing dictionary lookup. This prevents the earlier `KeyError` while making information loss explicit: decoding can recover the placeholder, but not the original unseen spelling.


In [19]:
# Reserve explicit IDs for document boundaries and unseen tokens.
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer, token in enumerate(all_tokens)}
print(len(vocab.items()))

1132


In [20]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [30]:
class SimpleTokenizerV2:
    """Map text to IDs with explicit unknown and document-boundary tokens.

    Args:
        vocab (dict[str, int]): Mapping that includes `<|unk|>` and
            `<|endoftext|>`.

    Attributes:
        str_to_int (dict[str, int]): Forward token-to-ID lookup.
        int_to_str (dict[int, str]): Reverse ID-to-token lookup.
    """

    def __init__(self, vocab):
        """Initialize forward and reverse vocabulary mappings."""
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        """Convert text into IDs, replacing unseen tokens with `<|unk|>`.

        Args:
            text (str): Text to split and encode.

        Returns:
            list[int]: Integer IDs for known and substituted tokens.
        """
        preprocessed = re.split(r'([,.:;?!_"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [item if item in self.str_to_int
                        else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        """Convert token IDs back into readable text.

        Args:
            ids (list[int]): Token IDs to decode.

        Returns:
            str: Reconstructed text with punctuation spacing normalized.

        Raises:
            KeyError: If an ID is absent from the reverse vocabulary.
        """
        text = " ".join(self.int_to_str[i] for i in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

## 7. Encode text with document boundaries

Two short passages are joined with `<|endoftext|>` and passed through the second tokenizer. The preserved IDs include both the boundary marker and unknown-token IDs. Decoding demonstrates the intended behavior: known words and punctuation are reconstructed, the boundary remains visible, and words outside the corpus vocabulary appear as `<|unk|>`.

This is a deliberately limited tokenizer. Later BPE-based tokenization improves coverage by representing unfamiliar words as smaller reusable pieces rather than collapsing each one to a single unknown marker.


In [31]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [32]:
tokenizer = SimpleTokenizerV2(vocab)
ids = tokenizer.encode(text)
print(ids)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [33]:
print(tokenizer.decode(ids))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


## Conclusion

This notebook exposes the essential vocabulary mechanics hidden behind higher-level tokenizer libraries. It progresses from corpus loading and regex-based splitting to reversible ID mappings, then uses a preserved failure to justify special-token handling.

| Version | Strength | Limitation demonstrated |
|---|---|---|
| `SimpleTokenizerV1` | Clear direct mapping between tokens and IDs | Raises `KeyError` for unseen tokens |
| `SimpleTokenizerV2` | Handles unseen tokens and document boundaries | Replaces unknown words with a generic marker |

The next tokenizer notebook uses GPT-2's byte pair encoding through `tiktoken`. BPE keeps the integer interface developed here while representing unfamiliar text with subword pieces, which reduces reliance on a single unknown token.


## Resources & References

- [Chapter 2 companion code — *Build a Large Language Model (From Scratch)*](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch02/01_main-chapter-code)
- [Python `re` documentation](https://docs.python.org/3/library/re.html)
- [OpenAI `tiktoken` repository and educational BPE notes](https://github.com/openai/tiktoken)
- [Book page — *Build a Large Language Model (From Scratch)*](https://www.manning.com/books/build-a-large-language-model-from-scratch)
